In [0]:
%sql
-- 1. Aseguramos que el schema existe
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- 2. Creamos la tabla con Liquid Clustering
-- Nota: CLUSTER BY se coloca antes de la consulta de origen
CREATE OR REPLACE TABLE workspace.gold.regional_sales_performance
CLUSTER BY (country, sales_date) 
AS 
SELECT 
    CASE 
        WHEN store_id LIKE 'GUA%' THEN 'Guatemala'
        WHEN store_id LIKE 'MX%' THEN 'México'
        ELSE 'Otros' 
    END AS country,
    store_id,
    date(transaction_time) AS sales_date,
    COUNT(DISTINCT ticket_id) AS total_transactions,
    SUM(quantity) AS total_items_sold,
    ROUND(SUM(total_sales_amount), 2) AS total_revenue
FROM workspace.silver.silver_sales
GROUP BY 1, 2, 3;

-- 3. (Opcional) Forzamos la optimización inmediata si es la primera carga masiva
OPTIMIZE workspace.gold.regional_sales_performance;